# NAS - Optuna

- **Authored by:** Matheus Ferreira Silva 
- **GitHub:**: https://github.com/MatheusFS-dev

## 1. Imports

In [1]:
import os
import shutil
import traceback

import optuna
from optuna.trial import TrialState

from IPython.display import clear_output as clear

# Araras Optuna utilities
from araras.optuna.analyze import analyze_study
from araras.optuna.utils import (
    cleanup_non_top_trials,
    get_top_trials,
    init_study_dirs,
    rename_top_k_files,
    save_top_k_trials,
)

# Utility functions
from araras.utils.misc import (
    clear,
)

## 2. Run Parameters 

In [2]:
TOP_K = 10 # Number of top trials to save

# True -> the greatest, the better
# False -> the least, the better
RANK_DESCENDING = True  

# Key to rank trials by:
# "value" -> objective trial value
# other e.g., "test_accuracy" -> user params
RANK_KEY = "test_accuracy_s009_full"

In [3]:
# Path to directory where the optuna_study dir is stored
STUDY_DIR = "/home/matheus/src/RayWise/results/v19/nas_cnn1d_flat_v19.2"

## Main

In [4]:
try:
    # ———————————————————————————————— Study Setup ——————————————————————————————— #
    # Initialize directories for the study
    (
        study_dir,
        args_dir,
        fig_dir,
        backup_dir,
        history_dir,
        model_dir,
        logs_dir,
    ) = init_study_dirs(STUDY_DIR)

    study = optuna.create_study(
        study_name=os.path.basename(study_dir),
        storage=f"sqlite:///{study_dir}/optuna_study.db",
        direction="minimize",
        pruner=optuna.pruners.HyperbandPruner(),
        load_if_exists=True,
    )

    # ——————————————————————— Processing the Study Results ——————————————————————— #
    top_trials = get_top_trials(
        study,
        top_k=TOP_K,
        rank_key=RANK_KEY,
        rank_descending=RANK_DESCENDING,
    )

    cleanup_paths = [
        (model_dir, "trial_{trial_id}.keras"),
        (fig_dir, "trial_{trial_id}.png"),
        (history_dir, "trial_{trial_id}.csv"),
    ]

    rename_paths = [
        (model_dir, ".keras"),
        (fig_dir, ".png"),
        (history_dir, ".csv"),
    ]

    extra_attrs = [
        "best_train_accuracy",
        "best_val_accuracy",
        "test_accuracy_s009",
        "test_accuracy_s009_full",
    ]

    save_top_k_trials(
        top_trials,
        args_dir=args_dir,
        study=study,
        extra_attrs=extra_attrs,
    )
    cleanup_non_top_trials(
        {t.number for t in study.trials},  # All trials
        {t.number for t in top_trials},  # Top trials ids
        cleanup_paths,
    )
    rename_top_k_files(top_trials, rename_paths)

    # ————————————————————————————— Log Trial Results ———————————————————————————— #
    with open(f"{study_dir}/trials.log", "w") as f:
        f.write(
            f"Total trials: {len(study.trials)}\n"
            f"Pruned trials: {sum(t.state==TrialState.PRUNED for t in study.trials)}\n"
            f"Failed trials: {sum(t.state==TrialState.FAIL for t in study.trials)}\n"
        )

    # —————————————————————————— Generate Study Analysis ————————————————————————— #
    (
        clear(),
        analyze_study(
            study,
            table_dir=os.path.join(study_dir, "analysis"),
            create_standalone=False,
            save_data=False,
            param_name_mapping=None,
            plots=[
                # "distributions",
                # "importances",
                # "correlations",
                # "boxplots",
                # "trends",
                # "ranges",
                # "contours",
                # "edf",
                # "intermediate",
                # "parallel_coordinate",
                # "rank",
                # "slice",
                # "history",
                # "timeline",
                "terminator",
            ],
        ),
    )
except Exception as e:
    print(f"\n An error occurred: {e}\n")
    traceback.print_exc()
finally:
    # Clean up directories
    shutil.rmtree(backup_dir, ignore_errors=True)
    if not os.listdir(logs_dir):
        os.rmdir(logs_dir)



Analyzing study...
--------------------------------------------------
Study info:
• Total trials: 512
• PRUNED trials: 380
• COMPLETE trials: 120
• RUNNING trials: 12
Parameter Template:
{
    "params_conv1d_0_act": "params_conv1d_0_act",
    "params_conv1d_0_filters": "params_conv1d_0_filters",
    "params_conv1d_1_act": "params_conv1d_1_act",
    "params_conv1d_1_filters": "params_conv1d_1_filters",
    "params_conv1d_2_act": "params_conv1d_2_act",
    "params_conv1d_2_filters": "params_conv1d_2_filters",
    "params_conv1d_2_kernel_size": "params_conv1d_2_kernel_size",
    "params_conv1d_3_act": "params_conv1d_3_act",
    "params_conv1d_3_filters": "params_conv1d_3_filters",
    "params_dense_0_act": "params_dense_0_act",
    "params_dense_0_dropout": "params_dense_0_dropout",
    "params_dense_0_units": "params_dense_0_units",
    "params_dense_1_act": "params_dense_1_act",
    "params_dense_1_dropout": "params_dense_1_dropout",
    "params_dense_1_units": "params_dense_1_units",

/home/matheus/src/araras/src/araras/optuna/plot_terminator_improvement.py:44: ExperimentalWarning: RegretBoundEvaluator is experimental (supported from v3.2.0). The interface can change in the future.
  improvement_evaluator = RegretBoundEvaluator()
/home/matheus/src/araras/src/araras/optuna/plot_terminator_improvement.py:49: ExperimentalWarning: CrossValidationErrorEvaluator is experimental (supported from v3.2.0). The interface can change in the future.
  error_evaluator = CrossValidationErrorEvaluator()


Variance of trials [20, 21, 22, 23, 24]: 7.576825529165258e-06
Variance of trials [21, 22, 23, 24, 25]: 6.791187269530685e-06
Variance of trials [22, 23, 24, 25, 26]: 9.526861071875606e-06
Variance of trials [23, 24, 25, 26, 27]: 3.652157707319274e-05
Variance of trials [24, 25, 26, 27, 28]: 3.17579799908856e-05
Variance of trials [25, 26, 27, 28, 29]: 2.7496447587738627e-05
Variance of trials [26, 27, 28, 29, 30]: 2.3804729087665608e-05
Variance of trials [27, 28, 29, 30, 31]: 2.4768522908017028e-05
Variance of trials [28, 29, 30, 31, 32]: 5.5023420192381966e-06
Variance of trials [29, 30, 31, 32, 33]: 4.1456228120760865e-05
Variance of trials [30, 31, 32, 33, 34]: 6.585073700871689e-05
Variance of trials [31, 32, 33, 34, 35]: 7.107621665089157e-05
Variance of trials [32, 33, 34, 35, 36]: 6.981671323114935e-05
Variance of trials [33, 34, 35, 36, 37]: 5.5116045060924805e-05
Variance of trials [34, 35, 36, 37, 38]: 5.060139086475419e-05
Variance of trials [35, 36, 37, 38, 39]: 8.5581982